# ML-4 : Evaluation des modeles (Python / sklearn)

**Navigation** : [Index](README.md) | [<< ML-3](ML-3-Entrainement-Python.ipynb) | [Suivant >>](ML-5-TimeSeries-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Evaluer** un modele de regression avec les metriques standard : R2, MAE, RMSE
2. **Valider** la robustesse via la **cross-validation K-fold** (plutot qu'un seul split train/test)
3. **Expliquer** un modele avec la **Permutation Feature Importance** (PFI)
4. **Comparer** systematiquement plusieurs algorithmes via `GridSearchCV` (equivalent AutoML)

### Parite ML.NET <-> Python

Ce notebook est le **jumeau Python** de [ML-4-Evaluation.ipynb](ML-4-Evaluation.ipynb) (ML.NET / C#). Il couvre les memes concepts (metriques, cross-validation, importance des variables) avec **scikit-learn** au lieu de `Microsoft.ML`.

| Concept ML.NET (C#) | Equivalent Python (sklearn) |
|---------------------|------------------------------|
| `mlContext.Regression.Evaluate` | `sklearn.metrics` (r2_score, mae, rmse) |
| `Regression.CrossValidate` | `sklearn.model_selection.cross_validate` |
| `PermutationFeatureImportance` | `sklearn.inspection.permutation_importance` |
| `mlContext.AutoML` (sweep) | `sklearn.model_selection.GridSearchCV` |
| Feature Contribution Calculation | (pas d'equivalent sklearn direct ; SHAP = extension) |

### Prerequis
- ML-2-Data&Features-Python et ML-3-Entrainement-Python completes
- Python 3.10+ avec `scikit-learn`, `pandas`, `numpy`

### Duree estimee : 45-60 minutes

---

## Preparation : charger les donnees et entrainer un modele de reference

Nous reutilisons le dataset **taxi-fare** (cf ML-2-Python). Objectif : predire `fare_amount`. Nous entrainons d'abord un modele de reference (regression lineaire), puis nous l'evaluons rigoureusement.

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# Dataset taxi-fare (in-memory, meme schema que ML-2-Python mais etendu pour la CV)
rng = np.random.RandomState(42)
n = 60
data = {
    "vendor_id":      rng.choice(["CMT", "VST"], n),
    "rate_code":      rng.choice([1.0, 2.0, 3.0], n).astype(float),
    "passenger_count":rng.choice([1.0, 2.0, 3.0, 4.0], n),
    "trip_time_in_secs": rng.randint(300, 3000, n).astype(float),
    "trip_distance":  rng.uniform(1.0, 12.0, n),
    "payment_type":   rng.choice(["CRD", "CSH"], n),
    "fare_amount":    None,
}
df = pd.DataFrame(data)
df["fare_amount"] = (
    2.5 + 2.2 * df["trip_distance"]
    + 0.004 * df["trip_time_in_secs"]
    + 1.5 * (df["vendor_id"] == "CMT").astype(float)
    + rng.normal(0, 1.5, n)
)
print("Dataset :", df.shape)
df.head()


Dataset : (60, 7)


,vendor_id,rate_code,passenger_count,trip_time_in_secs,trip_distance,payment_type,fare_amount
0,CMT,3.0,3.0,395.0,4.065111,CSH,14.337046
1,VST,3.0,3.0,2223.0,10.990925,CSH,36.682546
2,CMT,1.0,1.0,2057.0,3.635181,CSH,19.546704
3,CMT,1.0,4.0,2832.0,2.593844,CSH,22.200031
4,CMT,3.0,3.0,2578.0,6.383980,CSH,29.925114


### Split train / test

La regle d'or : **on n'evalue jamais un modele sur les donnees qui l'ont entraene**. On separe un jeu de test (20-30%) garde pour l'evaluation finale. En ML.NET : `mlContext.Data.TrainTestSplit`. En sklearn : `train_test_split`.

In [2]:
# Split train/test (80/20)
df_features = df.drop(columns=["fare_amount"])
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(df_features, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} lignes | Test: {X_test.shape[0]} lignes")


Train: 48 lignes | Test: 12 lignes


### Pipeline de preprocessing + modele de reference

On reutilise le `ColumnTransformer` du ML-2-Python (one-hot sur categorielles, imputer + scaler sur numeriques) chaine avec un `LinearRegression` (modele de reference simple).

In [3]:
# Pipeline preprocessing + LinearRegression (baseline)
categorical_cols = ["vendor_id", "payment_type"]
numeric_cols = ["rate_code", "passenger_count", "trip_time_in_secs", "trip_distance"]

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="mean")),
                      ("scaler", StandardScaler())]), numeric_cols),
])

baseline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])
baseline.fit(X_train, y_train)
print("Baseline (LinearRegression) entraene sur le train set.")


Baseline (LinearRegression) entraene sur le train set.


## Metriques d'evaluation : R2, MAE, RMSE

En ML.NET, `mlContext.Regression.Evaluate` retourne un objet `RegressionMetrics` avec `RSquared`, `MeanAbsoluteError`, `RootMeanSquaredError`. En sklearn, ces metriques sont des fonctions de `sklearn.metrics`.

### Definitions

- **R2 (coefficient de determination)** : proportion de variance expliquee par le modele. `R2=1` = parfait, `R2=0` = predit la moyenne, `R2<0` = pire que la moyenne. **A maximiser.**
- **MAE (Mean Absolute Error)** : erreur moyenne en valeur absolue, dans l'unite de la cible. **Interpretable directement.**
- **RMSE (Root Mean Squared Error)** : racine de l'erreur quadratique moyenne. Penalise davantage les grandes erreurs. **Plus sensible aux outliers que la MAE.**

In [4]:
# Evaluation sur le test set
y_pred = baseline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Baseline (LinearRegression) - metriques sur le TEST set :")
print(f"  R2   = {r2:.3f}   (proportion de variance expliquee)")
print(f"  MAE  = {mae:.2f} $  (erreur moyenne absolue)")
print(f"  RMSE = {rmse:.2f} $  (erreur quadratique moyenne)")
print()
print(f"Lecture : le modele explique {r2*100:.1f}% de la variance des tarifs.")
print(f"En moyenne, il se trompe de {mae:.2f} $ sur la prediction d'un tarif.")


Baseline (LinearRegression) - metriques sur le TEST set :
  R2   = 0.937   (proportion de variance expliquee)
  MAE  = 1.62 $  (erreur moyenne absolue)
  RMSE = 2.05 $  (erreur quadratique moyenne)

Lecture : le modele explique 93.7% de la variance des tarifs.
En moyenne, il se trompe de 1.62 $ sur la prediction d'un tarif.


## Cross-Validation K-fold : une evaluation plus robuste

Un seul split train/test est **bruite** : la metrique depend du tirage aleatoire. La **cross-validation K-fold** entraene et evalue le modele K fois sur des splits differents, puis on moyenne. C'est l'etalon-or pour estimer la performance **generalisable**.

En ML.NET : `Regression.CrossValidate`. En sklearn : `cross_validate`.

In [5]:
# Cross-validation 5-fold sur le TRAIN set
cv_results = cross_validate(baseline, X_train, y_train, cv=5,
                            scoring=["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"])
cv_r2 = cv_results["test_r2"]
cv_mae = -cv_results["test_neg_mean_absolute_error"]
cv_rmse = -cv_results["test_neg_root_mean_squared_error"]
print(f"Cross-validation 5-fold (sur le train set) :")
print(f"  R2   par fold: {[round(v,3) for v in cv_r2]}")
print(f"  R2   moyen = {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}")
print(f"  MAE  moyen = {cv_mae.mean():.2f} +/- {cv_mae.std():.2f} $")
print(f"  RMSE moyen = {cv_rmse.mean():.2f} +/- {cv_rmse.std():.2f} $")
print()
print(f"Lecture : le R2 moyen sur 5 folds ({cv_r2.mean():.3f}) est plus fiable qu'un seul split.")
print(f"Faible ecart-type ({cv_r2.std():.3f}) = modele stable across folds.")


Cross-validation 5-fold (sur le train set) :
  R2   par fold: [np.float64(0.984), np.float64(0.984), np.float64(0.981), np.float64(0.974), np.float64(0.899)]
  R2   moyen = 0.964 +/- 0.033
  MAE  moyen = 1.05 +/- 0.28 $
  RMSE moyen = 1.25 +/- 0.31 $

Lecture : le R2 moyen sur 5 folds (0.964) est plus fiable qu'un seul split.
Faible ecart-type (0.033) = modele stable across folds.


**Pourquoi la CV importe** : si le R2 sur un seul split etait `0.91` mais que la CV donne `0.85 +/- 0.06`, la CV revele que le modele est **moins robuste** qu'on ne le croyait. C'est la difference entre « j'ai eu de la chance sur ce split » et « le modele generalise bien ».

## Permutation Feature Importance (PFI)

La PFI repond a la question : **quelles features importent vraiment ?** Principe : pour chaque feature, on **melange** (permute) ses valeurs dans le jeu de test - cassant la relation avec la cible - et on mesure la **baisse de performance**. Une feature dont la permutation fait chuter le R2 est **importante** ; une feature dont la permutation ne change rien est **inutile**.

En ML.NET : `PermutationFeatureImportance`. En sklearn : `permutation_importance` (module `inspection`). Avantage majeur : **model-agnostic**.

In [6]:
# Permutation Feature Importance
pfi = permutation_importance(baseline, X_test, y_test, n_repeats=10,
                             random_state=42, scoring="r2")
pfi_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_moyenne": pfi.importances_mean,
    "ecart_type": pfi.importances_std,
}).sort_values("importance_moyenne", ascending=False).reset_index(drop=True)
print("Permutation Feature Importance (baseline, test set, scoring=R2) :")
print(pfi_df.to_string(index=False))
print()
print("Lecture : la feature en tete est la plus importante.")
print("Permuter cette colonne fait le plus chuter le R2 du modele.")


Permutation Feature Importance (baseline, test set, scoring=R2) :
          feature  importance_moyenne  ecart_type
    trip_distance            1.317343    0.232568
trip_time_in_secs            0.297095    0.121160
        vendor_id            0.006467    0.008514
  passenger_count           -0.001813    0.002349
        rate_code           -0.001836    0.002427
     payment_type           -0.003565    0.004212

Lecture : la feature en tete est la plus importante.
Permuter cette colonne fait le plus chuter le R2 du modele.


### Interpretation - lire les resultats de la PFI

La PFI donne un **classement** des features par impact sur la performance :

- **`trip_distance`** dominante : logique, le tarif d'un taxi depend principalement de la distance parcourue.
- **`trip_time_in_secs`** secondaire : la duree ajoute un signal (trafic, detour) au-dela de la distance.
- **`vendor_id`, `payment_type`, `rate_code`, `passenger_count`** faibles : la permutation ne degrade presque pas le modele.

**Insight cle** : la PFI est **conditionnelle** au modele. Une feature peut etre ignoree par un modele lineaire mais cruciale pour un RandomForest.

### Feature Contribution Calculation (FCC) - note de parite

Le notebook ML.NET couvre aussi la **Feature Contribution Calculation** (FCC), qui decompose la prediction **d'une instance** en contributions par feature (explicabilite locale). sklearn n'a pas d'equivalent natif direct ; l'ecosysteme Python utilise **SHAP** (`shap` library) ou **`partial_dependence`** pour cet angle. SHAP est hors scope de ce twin (dependance externe), mais le concept se transpose directement via SHAP en production.

## Comparer plusieurs algorithmes : GridSearchCV (equivalent AutoML)

En ML.NET, `mlContext.AutoML` balaie automatiquement l'espace des algorithmes et hyperparametres. En sklearn, `GridSearchCV` fait de meme de facon declarative : on fournit une grille d'hyperparametres, il teste toutes les combinaisons via cross-validation.

In [7]:
# GridSearchCV : comparer RandomForest hyperparametres via CV
param_grid = {
    "regressor__n_estimators": [50, 100],
    "regressor__max_depth": [3, 5, None],
}
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42)),
])
grid = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)
print(f"GridSearchCV - meilleure config : {grid.best_params_}")
print(f"Meilleur R2 (CV moyen sur train) : {grid.best_score_:.3f}")
print()
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
r2_best = r2_score(y_test, y_pred_best)
mae_best = mean_absolute_error(y_test, y_pred_best)
print(f"RandomForest optimise - metriques sur le TEST set :")
print(f"  R2   = {r2_best:.3f}   (vs baseline {r2:.3f})")
print(f"  MAE  = {mae_best:.2f} $  (vs baseline {mae:.2f})")
print(f"  Difference R2 : {r2_best - r2:+.3f}")


GridSearchCV - meilleure config : {'regressor__max_depth': 5, 'regressor__n_estimators': 50}
Meilleur R2 (CV moyen sur train) : 0.892

RandomForest optimise - metriques sur le TEST set :
  R2   = 0.863   (vs baseline 0.937)
  MAE  = 2.53 $  (vs baseline 1.62)
  Difference R2 : -0.075


## Comment puis-je ameliorer mon modele ?

Checklist d'amelioration (transversale ML.NET <-> sklearn) :

1. **Plus de donnees** : la performance plafonne souvent faute d'exemples.
2. **Feature engineering** : creer des features derivees (ex: `speed = trip_distance / trip_time`). Cf ML-2-Python.
3. **Regularisation** : `Ridge`/`Lasso` (sklearn) ou `L2Regularization` (ML.NET SDCA).
4. **Modeles plus expressifs** : GradientBoosting (`HistGradientBoostingRegressor`), ou `xgboost`/`lightgbm`.
5. **Hyperparametre tuning** : `GridSearchCV` ou `RandomizedSearchCV`.
6. **Cross-validation** : toujours valider par CV avant de conclure qu une amelioration est reelle.

## Exercices supplementaires

### Exercice 1 : Courbe ROC et calcul de l'AUC

**Objectif** : convertissez ce probleme de regression en classification binaire (tarif > mediane), puis tracez la courbe ROC et calculez l'AUC.

In [8]:
# Exercice 1 : Courbe ROC et calcul de l'AUC
# TODO etudiant : Convertissez en classification binaire (fare_amount > mediane) et evaluez avec ROC/AUC
# Indice : roc_auc_score, roc_curve de sklearn.metrics ; LogisticRegression comme classifieur
# Etape 1 : creez y_binary = (df["fare_amount"] > df["fare_amount"].median()).astype(int)
# Etape 2 : split + pipeline avec LogisticRegression au lieu de LinearRegression
# Etape 3 : calculez roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
# Etape 4 : tracez la courbe ROC via roc_curve + matplotlib

print("Exercice a completer : courbe ROC et AUC")
result_roc_auc = None  # TODO etudiant : remplacer par l'AUC


Exercice a completer : courbe ROC et AUC


### Exercice 2 : Analyse manuelle de la matrice de confusion

**Objectif** : pour la classification binaire de l'exercice 1, calculez la matrice de confusion et deduisez precision, rappel, F1 a la main.

In [9]:
# Exercice 2 : Analyse manuelle de la matrice de confusion
# TODO etudiant : Calculez la matrice de confusion et deduisez precision/rappel/F1 a la main
# Indice : confusion_matrix de sklearn.metrics ; comparez avec classification_report
# Etape 1 : predisez y_pred_class pour le modele binaire de l'exercice 1
# Etape 2 : calculez TP, FP, TN, FN manuellement
# Etape 3 : deduisez precision = TP/(TP+FP), rappel = TP/(TP+FN), F1 = 2*P*R/(P+R)
# Etape 4 : comparez avec classification_report de sklearn

print("Exercice a completer : matrice de confusion manuelle")
result_confusion_matrix = None  # TODO etudiant : remplacer par (TP, FP, TN, FN)


Exercice a completer : matrice de confusion manuelle


## Exercice : Analyse complete d'un modele de prediction

### Objectifs
- Construisez un pipeline complet sur un nouveau dataset (ex: `trip_duration` au lieu de `fare_amount`)
- Evaluez avec R2/MAE/RMSE + cross-validation
- Calculez la PFI et interpretez

### Indices
- Reutilisez `ColumnTransformer` + `Pipeline` du notebook
- `cross_validate` avec `cv=5` pour une estimation robuste
- `permutation_importance` pour l'explicabilite

In [10]:
# Exercice : Analyse complete d'un modele de prediction de duree de trajet
# Completez les parties TODO pour creer un pipeline d'evaluation complet

# TODO: definir la nouvelle cible y_dur = df["trip_time_in_secs"]
# TODO: split train/test
# TODO: construire un Pipeline (preprocessor + regressor au choix)
# TODO: fit + evaluer (R2, MAE, RMSE) sur test set
# TODO: cross-validation 5-fold
# TODO: permutation_importance et interpretation

print("Exercice a completer : analyse complete duree de trajet")
result_duration_analysis = None  # TODO etudiant : remplacer par le R2 final


Exercice a completer : analyse complete duree de trajet


## Resume

Dans ce notebook vous avez maitrise l'**evaluation rigoureuse** des modeles de regression avec scikit-learn, en miroir du notebook ML-4 .NET :

1. **Metriques** : R2 (variance expliquee), MAE (erreur moyenne interpretable), RMSE (sensible aux outliers). Trois angles complementaires, jamais un seul.
2. **Cross-validation K-fold** : estimer la performance **generalisable** plutot que de se fier a un seul split bruite.
3. **Permutation Feature Importance** : explicabilite **model-agnostic** - quelles features font vraiment bouger la performance.
4. **GridSearchCV** : comparer systematiquement algorithmes + hyperparametres via CV (equivalent AutoML ML.NET).

### Parite conceptuelle cle

L'evaluation rigoureuse suit le **meme cheminement** en ML.NET et sklearn : (1) split train/test, (2) entrainer un baseline, (3) metriques sur test, (4) cross-validation pour robustesse, (5) explicabilite (PFI), (6) sweep hyperparametres (AutoML / GridSearchCV). Les API different mais la **discipline methodologique** est identique.

### Prochaine etape

[ML-5-TimeSeries-Python.ipynb](ML-5-TimeSeries-Python.ipynb) aborde les **series temporelles** (equivalent SSA ML.NET).

## References

- [scikit-learn metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)
- [scikit-learn permutation_importance](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html)
- [scikit-learn GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- Notebook ML.NET source : [ML-4-Evaluation.ipynb](ML-4-Evaluation.ipynb)
- Epic parite .NET <-> Python : #4956
